# Step 3c: An "attack" answer, so handoffs reach the right team
### Neeraj, MMAI 894 (router fine-tuning)

**Why:** in the team's end-to-end test, the fine-tuned router lost mainly because it gives no reason for a handoff, so 7 attacks went to general advisors instead of the service manager. The team's grader decides the right team from two dataset columns: the request type, and whether the chat is an attack (`adversarial_type`). The second one is a real label (40 chats, 31 in our pool), so the model can learn it.

**What changes:** the model gets one more yes/no answer, **"is this chat an attack?"**, trained on that column. A fixed rule then turns type + attack into a reason (attack -> service manager, roadside -> roadside team, contested bill -> service manager...). Language comes from the word rule (no language head).

| Name | Training | Epochs | Answers |
|---|---|---|---|
| `full_attack` | ordinary fine-tuning, learning rate 2e-5 | 30 | type, escalation, attack |
| `lora_attack` | LoRA, learning rate 3e-4 (the setting that tied in notebook 03b) | 60 | type, escalation, attack |

Both on the two inputs that matter: the first message, and the full conversation (what the team's agent gives its router). Same folds and seeds as before. The model only ever reads the chat text; the attack column is the answer it learns from, never an input.

**What you need to do:** GPU on (Accelerator: GPU T4), Internet on, run all cells. The long cell takes about 55 to 65 minutes; if the session stops, run it again and it carries on. At the end, download the files listed in the last cell into `results/step03c`.

Everything shown is computed from saved files; this text holds no results.

## Step 0: Get the latest code and the LoRA library

In [ ]:
import importlib
import os
import shutil
import sys

REPO_URL = "https://github.com/neerajmehla28/northstar-finetuning"
REPO_DIR = "/kaggle/working/northstar-finetuning"

%cd /kaggle/working
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q {REPO_URL} {REPO_DIR}          # download the repo (public, so no login is needed)
!pip install -q peft                         # LoRA library
!pip uninstall -y -q torchao                 # Kaggle's older torchao makes the new peft refuse to run; we do not use it

%cd {REPO_DIR}
sys.path.insert(0, "src")
importlib.invalidate_caches()
for name in [m for m in sys.modules if m.startswith("northstar")]:
    del sys.modules[name]

assert os.path.exists("src/northstar/finetune/routing.py"), "code missing: has the latest code been pushed to GitHub?"
assert os.path.exists("results/cv_step03b/lora_3e4_60ep/first_message"), "notebook 03b's results are missing from the repo"
!git log --oneline -3

## Step 1: Check the machine and the data

In [ ]:
import torch
import pandas as pd
from northstar.finetune import protocol as P
from northstar.finetune.train_cv import TrainConfig, run_cv

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
assert torch.cuda.is_available(), "No GPU found. Set Accelerator to GPU T4 in the notebook settings."

df = P.load_all()                 # refuses to run if the dataset file is not the exact team file
pool = P.pool(df)                 # the 160 chats; the 40 test chats are not in here
print("Pool:", len(pool), "chats | attacks:", int(pool["attack_label"].sum()),
      "| attacks that need a person:", int(pool.loc[pool["attack_label"] == 1, "escalation_label"].sum()))

## Step 2: Settings

In [ ]:
OUT = "/kaggle/working/cv_step03c"
RESULTS = "/kaggle/working/results_03c"
os.makedirs(RESULTS, exist_ok=True)

VARIANTS = ("first_message", "full_conversation")
SEEDS = P.CV_SEEDS
HEADS = ("type", "escalation", "attack")

CONFIGS = {
    "full_attack": TrainConfig(epochs=30, heads=HEADS),                          # ordinary fine-tuning, the original recipe
    "lora_attack": TrainConfig(epochs=60, heads=HEADS, lora=True, lr=3e-4),      # LoRA as in notebook 03b
}
for name, c in CONFIGS.items():
    print(f"{name:12s} lora={c.lora} lr={c.lr} epochs={c.epochs} heads={c.heads}")

## Step 3: Smoke test
One fold, one seed, two epochs per model and input. The first one also downloads the model, so it is slower.

In [ ]:
import time

for name, cfg in CONFIGS.items():
    for v in VARIANTS:
        t0 = time.time()
        run_cv(cfg, v, f"/kaggle/working/smoke/{name}", folds=[0], seeds=[11], epochs=2, verbose=False)
        print(f"{name:12s} {v:17s} smoke test OK: {time.time() - t0:.0f}s for 2 epochs")
print("Expected full run: about 55 to 65 minutes (60 runs of about 40 to 75 s each).")

## Step 4: The full cross-validation
The long cell. Each run prints its time; rerun the cell if the session is interrupted (finished runs are kept).

In [ ]:
for name, cfg in CONFIGS.items():
    for v in VARIANTS:
        print(f"=== {name} / {v}")
        run_cv(cfg, v, f"{OUT}/{name}", seeds=SEEDS)
print("All runs finished.")

## Step 5: Results
Each model and input at its own best epoch (same rule as before). The attack answer is scored at the plain cut-off (a positive score means "attack"); notebook 05 chooses cut-offs and scores the handoff teams. "Without attack head" is the same model and input from earlier notebooks, to see whether the extra answer costs request-type accuracy.

In [ ]:
import numpy as np
from northstar.finetune import cv_analysis as A
from northstar.finetune import metrics as M

BEFORE = {("full_attack", "first_message"): "results/cv_old_recipe",
          ("full_attack", "full_conversation"): "results/cv_old_recipe",
          ("lora_attack", "first_message"): "results/cv_step03b/lora_3e4_60ep"}
y_type = pool["type_label"].values

rows, oof = [], {}
for name in CONFIGS:
    for v in VARIANTS:
        o = A.assemble(A.load_variant(f"{OUT}/{name}", v), pool, HEADS)
        curves = A.epoch_curves(o, pool)
        e_star, _ = A.pick_epoch(curves)
        s = A.summary_at_epoch(o, pool, e_star)
        oof[(name, v)] = (o, e_star, s)
        ps, at = s["per_seed"], s["ensemble_attack"]
        row = {"model": name, "input": v, "epoch": e_star,
               "type macro-F1": f"{ps['type_macro_f1'].mean():.3f} ± {ps['type_macro_f1'].std():.3f}",
               "escalation F2": f"{ps['esc_f2'].mean():.3f} ± {ps['esc_f2'].std():.3f}",
               "attack recall": f"{ps['attack_recall'].mean():.3f} ± {ps['attack_recall'].std():.3f}",
               "attack precision": f"{ps['attack_precision'].mean():.3f} ± {ps['attack_precision'].std():.3f}",
               "ensemble: attacks caught": f"{int(((s['ensemble_attack_logits'] > 0) & (pool['attack_label'].values == 1)).sum())} of {int(pool['attack_label'].sum())}",
               "ensemble: false attack alarms": at["false_alarms"]}
        if (name, v) in BEFORE:
            b = A.assemble(A.load_variant(BEFORE[(name, v)], v), pool)
            b_star, _ = A.pick_epoch(A.epoch_curves(b, pool))
            sb = A.summary_at_epoch(b, pool, b_star)
            ok_new = s["ensemble_type_logits"].argmax(-1) == y_type
            ok_old = sb["ensemble_type_logits"].argmax(-1) == y_type
            r = M.mcnemar_exact(ok_new, ok_old)
            row["type macro-F1 without attack head"] = f"{sb['per_seed']['type_macro_f1'].mean():.3f}"
            row["type: only new right / only old right (p)"] = f"{r['only_a_right']} / {r['only_b_right']} (p = {r['p_value']:.2f})"
        rows.append(row)
table = pd.DataFrame(rows).set_index(["model", "input"])
table.to_csv(f"{RESULTS}/attack_head_summary.csv")
table

## Step 6: Which attacks are missed?
Attack kinds in the dataset (`adversarial_type`) and how many of each the best-scoring setup catches. Helps explain where the model fails.

In [ ]:
best = max(oof, key=lambda k: oof[k][2]["ensemble_attack"]["f2"])
o, e_star, s = oof[best]
caught = s["ensemble_attack_logits"] > 0
kinds = pool.loc[pool["attack_label"] == 1, "adversarial_type"]
by_kind = pd.DataFrame({"attack kind": kinds.values, "caught": caught[pool["attack_label"].values == 1]})
kind_table = by_kind.groupby("attack kind")["caught"].agg(["sum", "count"]).rename(columns={"sum": "caught", "count": "chats"})
kind_table.to_csv(f"{RESULTS}/attacks_by_kind.csv")
print("Best setup by attack F2:", best, "| epoch", e_star)
kind_table

## Step 7: Save the results
Download `cv_step03c.zip` (raw scores of every run), the two tables, and the notebook itself (File -> Download Notebook) into `results/step03c`.

In [ ]:
shutil.make_archive(f"{RESULTS}/cv_step03c", "zip", OUT)
for name in sorted(os.listdir(RESULTS)):
    print(f"{name:40s} {os.path.getsize(os.path.join(RESULTS, name)) / 1024:8.0f} KB")

## What this notebook shows, and its limits
- **Shows:** whether the model can learn "is this an attack?" from the chat text, on which input, with which training method, and whether the extra answer costs request-type accuracy.
- **Limits:**
  - Only 31 attack chats to learn from, in 5 kinds of 5 to 7 chats each (prompt injection, personal-data extraction, policy pressure, warranty fraud, manufacturer impersonation).
  - The attack cut-off here is the plain one; notebook 05 chooses it on the pool, like the escalation cut-off.
  - The team's input check (word patterns) also catches some attacks in the real pipeline; that is not included here, and it helps every router equally.
  - Synthetic chats from one generator; the fresh set in the final run is the independent check.